# Practice & Assignment — L02 Probability & Statistics for ML

**This notebook has two parts:**

| Part | What it is | When to do it |
|---|---|---|
| **Practice** (Tiers 1–3) | Guided → partial → open exercises | During or just after the lesson session |
| **Assignment** (Exercises 1–3) | Independent application in a hospital scenario | After completing Practice |

Work through both parts in order. Sample solutions for all exercises are at the bottom
— check them only after you have attempted each exercise yourself.

**Scenario thread:** All practice exercises follow **Tom Bradley** at **Lakeside Bank**.
The assignment exercises move to a **hospital satisfaction survey** so you practise
transferring the concepts to a completely new context.
*(Sarah's been on a hospital project before — in L01's assignment — so the setting
will feel familiar even though the questions are different.)*


In [ ]:
# Setup — run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print("✅ Libraries loaded — ready to go!")

---
# PART A — PRACTICE (Tiers 1–3)

## Tom Bradley's Problem

**Tom Bradley** is Head of Analytics at Lakeside Bank — NorthStar Retail's banking partner.
Sarah consulted here during the L01 assignment.

Lakeside recently launched a new **mobile-app onboarding flow** for first-time customers.
The old flow was a 12-step web form. The new flow is a guided 5-step mobile wizard.

Tom asks:

> *"The new flow took 6 months to build. I need to know whether it's actually reducing*
> *customer complaints in the first 30 days. Show me the data the right way."*

Tom's team collected data for 500 customers on the old flow (control) and
500 customers on the new flow (treatment) over the same 8-week period.


In [ ]:
# Generate Tom's dataset (run this cell — do not change)
np.random.seed(42)

N_OLD = 500  # customers on old 12-step web form
N_NEW = 500  # customers on new 5-step mobile wizard

# Old flow: relatively high complaint rate (18%) and right-skewed resolution times
old_complaint_rate = 0.18
old_complaints     = np.random.binomial(1, old_complaint_rate, size=N_OLD)
old_wait_days      = np.random.exponential(scale=8, size=N_OLD) + 1   # days to resolve

# New flow: lower complaint rate (10%) and shorter resolution times
new_complaint_rate = 0.10
new_complaints     = np.random.binomial(1, new_complaint_rate, size=N_NEW)
new_wait_days      = np.random.exponential(scale=5, size=N_NEW) + 1

# Time spent on onboarding (minutes) — old flow has a long right tail of stuck users
old_onboard_minutes = np.random.lognormal(mean=np.log(18), sigma=0.4, size=N_OLD).clip(5, 90)
new_onboard_minutes = np.random.lognormal(mean=np.log(10), sigma=0.25, size=N_NEW).clip(3, 30)

print(f"Old flow customers: {N_OLD}")
print(f"New flow customers: {N_NEW}")
print(f"Old flow complaints: {old_complaints.sum()} ({old_complaints.mean():.1%})")
print(f"New flow complaints: {new_complaints.sum()} ({new_complaints.mean():.1%})")
print()
print("Data loaded. Proceed to Tier 1.")

---
## 🟢 Tier 1 — Guided Example

Read through this worked example, run each cell, and make sure you understand what
each step is doing before moving on to Tier 2.

### Task: Describe the distribution of onboarding time for each flow


In [ ]:
# Tier 1 — Fully worked example
# Compare the distribution of onboarding time for old vs new flow

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (data, label, color) in enumerate([
        (old_onboard_minutes, 'Old flow (12-step web form)', 'steelblue'),
        (new_onboard_minutes, 'New flow (5-step mobile wizard)', 'coral')
    ]):
    ax = axes[i]
    ax.hist(data, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(), color='orange', linewidth=2.5, label=f"Mean: {data.mean():.1f} min")
    ax.axvline(np.median(data), color='green', linewidth=2.5, linestyle='--',
               label=f"Median: {np.median(data):.1f} min")
    ax.set_xlabel("Onboarding time (minutes)")
    ax.set_ylabel("Customers")
    ax.set_title(label)
    ax.legend()

plt.suptitle("Distribution of Onboarding Time — Old vs New Flow", fontweight='bold')
plt.tight_layout()
plt.show()

# Summary statistics
print("=== Summary Statistics ===")
for data, label in [(old_onboard_minutes, "Old flow"), (new_onboard_minutes, "New flow")]:
    print(f"  {label}:")
    print(f"    Mean:   {data.mean():.1f} min")
    print(f"    Median: {np.median(data):.1f} min")
    print(f"    Std:    {data.std():.1f} min")
    print(f"    Skew:   {pd.Series(data).skew():.2f}")
    print()


In [ ]:
# Tier 1 continued — Z-score to flag unusually long onboarding sessions
old_z_scores = (old_onboard_minutes - old_onboard_minutes.mean()) / old_onboard_minutes.std()
new_z_scores = (new_onboard_minutes - new_onboard_minutes.mean()) / new_onboard_minutes.std()

old_outliers = (np.abs(old_z_scores) > 3).sum()
new_outliers = (np.abs(new_z_scores) > 3).sum()

print("Onboarding sessions with |Z| > 3 (unusually long or short):")
print(f"  Old flow: {old_outliers} sessions ({old_outliers/N_OLD:.1%})")
print(f"  New flow: {new_outliers} sessions ({new_outliers/N_NEW:.1%})")
print()
print("Interpretation:")
print("  The old flow has a wider spread (higher std). Longer right tail = right-skewed.")
print("  The new flow is more concentrated — shorter and more consistent.")
print("  A few old-flow sessions ran very long (high Z-score): these are candidates")
print("  for investigation (technical errors? confused customers?).")


**Tier 1 Reflection** *(write your answers here — double-click to edit)*

1. Which flow has a more normal-looking distribution of onboarding times? How can you tell?
2. Why is the median a more honest summary of "typical onboarding time" than the mean for the old flow?
3. Tom says "we saved an average of about 9 minutes per customer." Is that a fair way to communicate the improvement? What else would you add?

> **Sample answers:**
> 1. The new flow is more symmetric (lower skew value, mean ≈ median). The old flow has a longer right tail — some customers spent much more than the average, pulling the mean up.
> 2. The old flow is right-skewed — a handful of very long sessions pull the mean upward. The median better represents what a typical customer experienced, since it isn't affected by those outliers.
> 3. Technically accurate but incomplete. A more complete statement would include: "The median time dropped from about 18 minutes to 10 minutes (~44% reduction), and the new flow is also more consistent — fewer customers hit unusually long sessions." Always pair the mean with the median and some measure of spread for skewed data.

---
## 🟡 Tier 2 — Partial Exercises

The code structure is provided. Fill in the blanks where you see `___`.
**Each cell runs without errors before you fill anything in** — the `___` are in comments,
not inside function calls.

### Task 2A: Compute and visualise the 95% confidence interval for each group's complaint rate


In [ ]:
# Tier 2A — Fill in the blanks

# Step 1: Calculate sample complaint rates
old_rate = old_complaints.mean()   # proportion of old-flow customers who complained
new_rate = new_complaints.mean()   # proportion of new-flow customers who complained

print(f"Old flow complaint rate: {old_rate:.3f}")
print(f"New flow complaint rate: {new_rate:.3f}")
print()

# Step 2: Compute the margin of error for each group's 95% CI
# Formula: margin = 1.96 * sqrt(p * (1-p) / n)
z_95 = 1.96

old_se = None   # replace None with: np.sqrt(old_rate * (1 - old_rate) / N_OLD)
new_se = None   # replace None with: np.sqrt(new_rate * (1 - new_rate) / N_NEW)

old_margin = None   # replace None with: z_95 * old_se
new_margin = None   # replace None with: z_95 * new_se

# Step 3: Compute lower and upper bounds
old_lower = None   # replace None with: old_rate - old_margin
old_upper = None   # replace None with: old_rate + old_margin
new_lower = None   # replace None with: new_rate - new_margin
new_upper = None   # replace None with: new_rate + new_margin

# The if-blocks below will print results only after you fill in the values above
if old_se is not None:
    print(f"Old flow 95% CI: [{old_lower:.3f}, {old_upper:.3f}]  (i.e. {old_lower:.1%} to {old_upper:.1%})")
    print(f"New flow 95% CI: [{new_lower:.3f}, {new_upper:.3f}]  (i.e. {new_lower:.1%} to {new_upper:.1%})")
else:
    print("Fill in the blanks above to see the confidence intervals.")


<details>
<summary>💡 Hints — click to reveal only if stuck</summary>

**Standard error of a proportion:**
```python
old_se = np.sqrt(old_rate * (1 - old_rate) / N_OLD)
new_se = np.sqrt(new_rate * (1 - new_rate) / N_NEW)
```

**Margin of error (multiply SE by Z for 95%):**
```python
old_margin = z_95 * old_se
new_margin = z_95 * new_se
```

**Lower and upper bounds:**
```python
old_lower = old_rate - old_margin
old_upper = old_rate + old_margin
```
</details>


In [ ]:
# Tier 2B — Fill in the visualisation of the CIs
# This cell creates an error-bar plot showing both CIs
# Only meaningful after 2A is filled in

if old_se is not None:
    fig, ax = plt.subplots(figsize=(8, 5))

    for i, (rate, lower, upper, label, color) in enumerate([
        (old_rate, old_lower, old_upper, 'Old flow', 'steelblue'),
        (new_rate, new_lower, new_upper, 'New flow', 'coral')
    ]):
        ax.errorbar(x=i, y=rate,
                    yerr=[[rate - lower], [upper - rate]],
                    fmt='o', color=color, capsize=10, capthick=2,
                    markersize=12, linewidth=2,
                    label=f'{label}: {rate:.1%} (95% CI: {lower:.1%}–{upper:.1%})')

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Old flow', 'New flow'])
    ax.set_ylabel('30-day Complaint Rate')
    ax.set_title("Complaint Rates with 95% Confidence Intervals\nTom's Onboarding Flow Comparison")
    ax.legend(loc='upper right')
    ax.set_ylim(0, 0.30)
    plt.tight_layout()
    plt.show()

    # Interpretation prompt
    print("Do the two confidence intervals overlap?")
    overlapping = old_lower <= new_upper and new_lower <= old_upper
    print(f"  → {'Yes, they overlap' if overlapping else 'No, they do not overlap'}")
    print()
    print("Overlapping CIs suggest the difference might not be statistically significant.")
    print("Non-overlapping CIs strongly suggest a real difference.")
    print("We'll run a proper test in Tier 3 to get the p-value.")
else:
    print("Fill in Tier 2A first.")


**Tier 2 Reflection** *(write your answers here)*

1. Do the confidence intervals for old and new flow overlap? What does that suggest about the significance of the difference?
2. If Tom wanted to narrow the confidence intervals, what would he need to do? What's the practical tradeoff?
3. Tom says: "The new flow clearly works — the complaint rate dropped from 18% to 13%!" What nuance is he missing?

> **Sample answers:**
> 1. If the CIs overlap slightly, it does not automatically mean the difference is statistically significant — but it's a warning sign to run the formal test. Non-overlapping CIs are a strong visual signal of significance.
> 2. He would need more customers per group. The CI width scales as 1/√n, so to halve the width he'd need 4× more customers. The practical tradeoff: more data means a longer wait before making the decision.
> 3. He's reporting point estimates without uncertainty. A more complete statement: "The complaint rate dropped from 18% to 13% — but given the sample sizes, each rate has about ±3–4 percentage points of uncertainty. We need a formal test to confirm the difference is statistically significant rather than due to random variation."


---
## 🔴 Tier 3 — Open Exercise

No code is provided. Use what you've learned to answer Tom's original question.

### Task: Run an A/B test to determine whether the new onboarding flow significantly reduces complaint rates. Report the result in plain English for Tom.

**Success criteria:**
- Calculate the p-value using a two-proportion z-test (formula in `lesson.md`)
- Clearly state what the null and alternative hypotheses are
- State whether the result is statistically significant at the 0.05 level
- Report the *effect size* (the actual difference in rates), not just significance
- Write a 2–3 sentence plain-English summary for Tom (non-technical)

In [ ]:
# Tier 3 — Write your code here
# Hint: the two-proportion z-test from lesson.md, in code:
#   p_pool = (old_complaints.sum() + new_complaints.sum()) / (N_OLD + N_NEW)
#   se     = np.sqrt(p_pool * (1 - p_pool) * (1/N_OLD + 1/N_NEW))
#   z      = (new_complaints.mean() - old_complaints.mean()) / se
#   p_one_sided = st.norm.cdf(z)   # one-sided: new < old
#
# Replace the placeholders below with your computation.

---
# PART B — ASSIGNMENT (Independent Exercises)

## The Hospital Satisfaction Survey

*(Sarah's been on a hospital project before — in the L01 assignment, she analysed
readmission rates at the same hospital. Now the hospital analytics team is back,
this time with a patient satisfaction question.)*

**Scenario:** St. Cecilia's Hospital has been running a quarterly patient satisfaction
survey for 3 years. The analytics team wants to know three things:
1. What does the distribution of satisfaction scores look like — and is it the right shape for the reporting they've been doing?
2. Is the hospital's 78% "highly satisfied" rate a reliable estimate, or could it be misleading?
3. A new "bedside communication training" programme was rolled out to half the wards last quarter. Did it improve satisfaction?

You have access to three months of survey data from two ward groups.


In [ ]:
# Assignment data setup — run this cell, do not modify
np.random.seed(99)

# Satisfaction scores: 1 (very dissatisfied) to 10 (very satisfied)
# Old wards (no new training): slightly left-skewed (most patients fairly happy)
n_old_wards   = 320
n_new_training = 280

old_ward_scores = np.clip(
    np.random.normal(loc=7.2, scale=1.8, size=n_old_wards).round(), 1, 10
).astype(int)

# New training wards: slightly higher mean, tighter spread
new_training_scores = np.clip(
    np.random.normal(loc=7.8, scale=1.4, size=n_new_training).round(), 1, 10
).astype(int)

# Binary "highly satisfied" (score >= 8)
old_high_sat   = (old_ward_scores   >= 8).astype(int)
new_high_sat   = (new_training_scores >= 8).astype(int)

print(f"Old wards (control):        {n_old_wards} patients, mean score: {old_ward_scores.mean():.2f}")
print(f"New training wards:         {n_new_training} patients, mean score: {new_training_scores.mean():.2f}")
print(f"Old wards 'highly satisfied' rate:     {old_high_sat.mean():.1%}")
print(f"New training 'highly satisfied' rate:  {new_high_sat.mean():.1%}")


---
### Exercise 1 — Distribution Analysis

**Business question:** The hospital currently reports the *mean* satisfaction score to its board.
Is this a fair summary? What would you recommend instead?

**Your tasks:**
1. Plot the distribution of satisfaction scores for both ward groups (histograms side by side)
2. Compute mean, median, and standard deviation for each group
3. Calculate the skewness for each group using `pd.Series(data).skew()`
4. State whether you would recommend reporting the mean or median to the board, and why

**Write your interpretation in a markdown cell** after the code — not as a print statement.


In [ ]:
# Exercise 1 — Your code here



*Write your interpretation here (double-click to edit):*


---
### Exercise 2 — Confidence Interval

**Business question:** The hospital CEO announced in the annual report that "78% of patients are highly satisfied." How reliable is this figure?

**Your tasks:**
1. Compute the overall "highly satisfied" rate across all wards (combine both groups as if it's one pool: `np.concatenate([old_high_sat, new_high_sat])`)
2. Calculate the 95% confidence interval for this proportion using the formula from the lesson
3. Plot an error-bar chart showing the estimate and its CI
4. Write 2–3 sentences interpreting the CI for the CEO — in plain English, no statistical jargon

**Important:** State clearly whether the CEO's claim of "78%" is consistent with your CI.


In [ ]:
# Exercise 2 — Your code here



*Write your interpretation here (double-click to edit):*


---
### Exercise 3 — A/B Test

**Business question:** Did the communication training programme improve the "highly satisfied" rate in the wards that received it?

**Your tasks:**
1. Define the null and alternative hypotheses in plain English
2. Run a two-proportion z-test (formula in `lesson.md`) comparing old wards vs new-training wards on the "highly satisfied" binary outcome
3. Report the p-value and state whether the result is statistically significant at 0.05
4. Report the observed effect size (the difference in rates in percentage points)
5. Address the three mis-readings in your write-up:
   - Correctly state what the p-value means
   - Comment on whether the effect is both statistically AND practically significant
   - If the result were p = 0.09, how would your conclusion change?

**Write your interpretation in the markdown cell below the code.**

In [ ]:
# Exercise 3 — Your code here



*Write your interpretation here (double-click to edit):*


---
## Submission Checklist

Before submitting, confirm:

- [ ] Exercise 1: Histogram of both ward groups is plotted; mean, median, std computed; clear recommendation stated
- [ ] Exercise 2: Combined "highly satisfied" rate and its 95% CI computed; CEO's "78%" claim addressed
- [ ] Exercise 3: Null and alternative hypotheses stated; p-value computed; effect size reported; all three mis-readings addressed
- [ ] Each exercise has a written interpretation in plain English in the markdown cell (not just printed output)
- [ ] All code cells run without errors when executed top to bottom
- [ ] Variable names are clear; at least one comment per logical block

---
---
# SAMPLE SOLUTIONS
## (Check only after attempting each exercise yourself)
---


### Solution: Tier 2A — Confidence Intervals for Complaint Rates


In [ ]:
# Solution: Tier 2A
old_rate = old_complaints.mean()
new_rate = new_complaints.mean()
z_95     = 1.96

old_se     = np.sqrt(old_rate * (1 - old_rate) / N_OLD)
new_se     = np.sqrt(new_rate * (1 - new_rate) / N_NEW)

old_margin = z_95 * old_se
new_margin = z_95 * new_se

old_lower  = old_rate - old_margin
old_upper  = old_rate + old_margin
new_lower  = new_rate - new_margin
new_upper  = new_rate + new_margin

print("=== Solution: Tier 2A ===")
print(f"Old flow: {old_rate:.1%}  (95% CI: {old_lower:.1%}–{old_upper:.1%})")
print(f"New flow: {new_rate:.1%}  (95% CI: {new_lower:.1%}–{new_upper:.1%})")


### Solution: Tier 3 — A/B Test for Tom


In [ ]:
# Solution: Tier 3
# Null hypothesis:        The new onboarding flow has no effect on 30-day complaint rate
# Alternative hypothesis: The new onboarding flow reduces the 30-day complaint rate

# Two-proportion z-test (computed from first principles — see lesson.md)
old_n = old_complaints.sum()
new_n = new_complaints.sum()
p_pool = (old_n + new_n) / (N_OLD + N_NEW)
se     = np.sqrt(p_pool * (1 - p_pool) * (1/N_OLD + 1/N_NEW))
z_stat = (new_complaints.mean() - old_complaints.mean()) / se
p_value = st.norm.cdf(z_stat)   # one-sided: new < old

effect_size = new_complaints.mean() - old_complaints.mean()

print("=== Solution: Tier 3 A/B Test ===")
print(f"Old flow complaint rate: {old_complaints.mean():.1%}")
print(f"New flow complaint rate: {new_complaints.mean():.1%}")
print(f"Observed difference:     {effect_size:+.1%} ({effect_size*100:+.1f} percentage points)")
print(f"Z-statistic:             {z_stat:.3f}")
print(f"P-value (one-sided):     {p_value:.4f}")
print()
if p_value < 0.05:
    print("Result: STATISTICALLY SIGNIFICANT (p < 0.05)")
    print("We reject the null hypothesis.")
else:
    print("Result: Not statistically significant (p >= 0.05)")

print()
print("Plain-English summary for Tom:")
print(f"  The new onboarding flow reduced 30-day complaint rates by approximately")
print(f"  {abs(effect_size)*100:.1f} percentage points ({old_complaints.mean():.0%} → {new_complaints.mean():.0%}).")
print(f"  This difference is statistically significant (p = {p_value:.3f}), meaning it is")
print(f"  unlikely to be due to random variation. The data supports rolling out the new")
print(f"  flow more broadly — subject to cost and operational considerations.")

### Solution: Exercise 1 — Distribution Analysis


In [ ]:
# Solution: Exercise 1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, label, color in [
    (axes[0], old_ward_scores,      'Old wards (control)',      'steelblue'),
    (axes[1], new_training_scores,  'New training wards',       'coral')
]:
    ax.hist(data, bins=range(1, 12), align='left', color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(),       color='orange', linewidth=2.5, label=f"Mean: {data.mean():.2f}")
    ax.axvline(np.median(data),   color='green',  linewidth=2.5, linestyle='--',
               label=f"Median: {np.median(data):.0f}")
    ax.set_xlabel("Satisfaction Score (1–10)")
    ax.set_ylabel("Number of Patients")
    ax.set_title(label)
    ax.legend()

plt.suptitle("Distribution of Patient Satisfaction Scores", fontweight='bold')
plt.tight_layout()
plt.show()

for data, label in [(old_ward_scores, "Old wards"), (new_training_scores, "New training")]:
    print(f"{label}: mean={data.mean():.2f}, median={np.median(data):.0f}, "
          f"std={data.std():.2f}, skew={pd.Series(data).skew():.2f}")

print()
print("Interpretation:")
print("  Both distributions are slightly left-skewed (mean < median would indicate this;")
print("  negative skew means more scores cluster toward the high end with a tail toward 1–2).")
print("  Reporting the mean to the board is reasonable but should be accompanied by the")
print("  median — since a few very dissatisfied patients (score 1–2) pull the mean down.")
print("  Reporting score distributions by category (1–4: dissatisfied, 5–7: neutral, 8–10: satisfied)")
print("  is even more informative than a single number.")


### Solution: Exercise 2 — Confidence Interval for CEO Claim


In [ ]:
# Solution: Exercise 2
all_high_sat  = np.concatenate([old_high_sat, new_high_sat])
n_total       = len(all_high_sat)
overall_rate  = all_high_sat.mean()

se_overall    = np.sqrt(overall_rate * (1 - overall_rate) / n_total)
margin        = 1.96 * se_overall
ci_low        = overall_rate - margin
ci_high       = overall_rate + margin

print(f"Overall 'highly satisfied' rate: {overall_rate:.1%}")
print(f"95% Confidence Interval: [{ci_low:.1%}, {ci_high:.1%}]")
print(f"Margin of error: ±{margin:.1%}")
print()
print("Plain-English interpretation for the CEO:")
print(f"  The survey of {n_total} patients found {overall_rate:.0%} rated themselves 'highly satisfied'")
print(f"  (score 8, 9, or 10). Given our sample size, the true rate is likely between")
print(f"  {ci_low:.0%} and {ci_high:.0%}. The CEO's statement of 78% is consistent with")
print(f"  this interval {'✅' if ci_low <= 0.78 <= ci_high else '❌ — review the exact numbers'}.")
print()
print("  Recommended phrasing for the annual report:")
print(f"  'Approximately {overall_rate:.0%} of surveyed patients reported high satisfaction")
print(f"  (95% CI: {ci_low:.0%}–{ci_high:.0%}), based on {n_total} responses.'")


### Solution: Exercise 3 — A/B Test for Hospital Training


In [ ]:
# Solution: Exercise 3
# Null hypothesis (H₀):        The communication training has no effect on the 'highly satisfied' rate
# Alternative hypothesis (H₁): The training increases the 'highly satisfied' rate

# Two-proportion z-test (one-sided: new > old, i.e. training improves satisfaction)
old_n = old_high_sat.sum()
new_n = new_high_sat.sum()
p_pool_ex3 = (old_n + new_n) / (n_old_wards + n_new_training)
se_ex3     = np.sqrt(p_pool_ex3 * (1 - p_pool_ex3) * (1/n_old_wards + 1/n_new_training))
z_stat_ex3 = (new_high_sat.mean() - old_high_sat.mean()) / se_ex3
p_val_ex3  = 1 - st.norm.cdf(z_stat_ex3)   # one-sided: new > old

effect_ex3 = new_high_sat.mean() - old_high_sat.mean()

print("=== Solution: Exercise 3 ===")
print(f"Old wards 'highly satisfied': {old_high_sat.mean():.1%}")
print(f"New training 'highly satisfied': {new_high_sat.mean():.1%}")
print(f"Observed difference: {effect_ex3:+.1%}")
print(f"Z-statistic: {z_stat_ex3:.3f}")
print(f"P-value (one-sided): {p_val_ex3:.4f}")
print()
sig = p_val_ex3 < 0.05
print(f"Statistically significant at 0.05: {'YES' if sig else 'NO'}")
print()
print("Addressing the three mis-readings:")
print()
print(f"1. The p-value ({p_val_ex3:.4f}) means: if the training had zero effect, we would see a")
print(f"   difference this large or larger {p_val_ex3*100:.1f}% of the time by chance alone.")
print(f"   It does NOT mean there is a {p_val_ex3*100:.1f}% chance the training had no effect.")
print()
print(f"2. The effect is {effect_ex3*100:.1f} percentage points. Whether this is practically")
print(f"   significant depends on context — e.g., how much did the training cost?")
print(f"   A 5pp improvement on hundreds of patients per quarter may well justify the investment.")
print()
print("3. If the p-value were 0.09: we would say 'insufficient evidence to reject the null")
print("   hypothesis at the 0.05 level.' This does NOT mean the training had no effect —")
print("   it may mean the sample was too small. Recommendation: extend the trial period.")